Primeiro passo: instalação do ambiente

In [1]:
%pip install sqlite

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement sqlite (from versions: none)
ERROR: No matching distribution found for sqlite


Segundo passo: criação de base de dados local.

In [2]:
import sqlite3
import pandas as pd

queries = {"query_hospital": 
"""
CREATE TABLE IF NOT EXISTS hospital(
    CNES TEXT PRIMARY KEY,
    NATUREZA TEXT,
    GESTAO TEXT,
    NAT_JUR TEXT
);
""",
"query_cid10": 
"""
CREATE TABLE IF NOT EXISTS cid10(
    CID TEXT PRIMARY KEY,
    CD_DESCRICAO TEXT
);
""",
"query_procedimentos":"""
CREATE TABLE IF NOT EXISTS procedimentos(
    PROC_REA TEXT PRIMARY KEY,
    NOME_PROC TEXT
);
""",
"query_municipios":"""
CREATE TABLE IF NOT EXISTS municipios(
    codigo_6d TEXT PRIMARY KEY,
    codigo_ibge TEXT,
    nome TEXT,
    latitude REAL,
    longitude REAL,
    estado TEXT
);
""",
"query_dados_ibge":"""
CREATE TABLE IF NOT EXISTS dado_ibge(
    codigo_municipio_completo TEXT PRIMARY KEY,
    nome_municipio TEXT,
    sigla_estado TEXT,
    populacao REAL,
    densidade_demografica REAL,
    salario_medio REAL,
    mortalidade_infantil REAL,

    FOREIGN KEY (codigo_municipio_completo) REFERENCES municipios (codigo_ibge)
);
""",
"query_internacoes":"""
CREATE TABLE IF NOT EXISTS internacoes(
    N_AIH TEXT PRIMARY KEY,
    CNES TEXT,
    DT_INTER TEXT,
    DT_SAIDA TEXT,
    QT_DIARIAS INTEGER,
    PROC_REA TEXT,
    VAL_TOT REAL,
    DIAG_PRINC TEXT,
    DIAG_SECUN TEXT,
    NASC TEXT,
    SEXO INTEGER,
    IDADE REAL,
    MUNIC_RES TEXT,

    FOREIGN KEY (CNES) REFERENCES hospital (CNES),
    FOREIGN KEY (DIAG_PRINC) REFERENCES cid10 (CID),
    FOREIGN KEY (PROC_REA) REFERENCES procedimentos (PROC_REA),
    FOREIGN KEY (MUNIC_RES) REFERENCES municipios (codigo_6d)
);""",
"query_mortes":"""CREATE TABLE IF NOT EXISTS mortes(
    N_AIH TEXT PRIMARY KEY,
    CID_MORTE TEXT,

    FOREIGN KEY (N_AIH) REFERENCES internacoes (N_AIH),
    FOREIGN KEY (CID_MORTE) REFERENCES cid10 (CID)
);""",
"query_condicoes_especificas":"""CREATE TABLE IF NOT EXISTS condicoes_especificas(
    N_AIH TEXT PRIMARY KEY,
    IND_VDRL TEXT,

    FOREIGN KEY (N_AIH) REFERENCES internacoes (N_AIH)
);"""
}
with sqlite3.connect("database.db") as db:
    cursor = db.cursor()
    cursor.execute("PRAGMA foreign_keys = ON;")
    for querie in queries.values():
        cursor.execute(querie)


In [3]:
import sqlite3

insert_queries = [
    # hospital
    "INSERT OR IGNORE INTO hospital (CNES, NATUREZA, GESTAO, NAT_JUR) VALUES ('1234567', 'Público', 'Estadual', 'Administração Direta');",
    "INSERT OR IGNORE INTO hospital (CNES, NATUREZA, GESTAO, NAT_JUR) VALUES ('7654321', 'Privado', 'Municipal', 'Empresa Privada');",
    "INSERT OR IGNORE INTO hospital (CNES, NATUREZA, GESTAO, NAT_JUR) VALUES ('1112223', 'Filantrópico', 'Estadual', 'Associação');",
    "INSERT OR IGNORE INTO hospital (CNES, NATUREZA, GESTAO, NAT_JUR) VALUES ('2234416', 'Público', 'Municipal', 'Administração Direta');", # Para consulta A4 e B6
    "INSERT OR IGNORE INTO hospital (CNES, NATUREZA, GESTAO, NAT_JUR) VALUES ('9999999', 'Privado', 'Estadual', 'Empresa Privada');",
    
    # cid10
    "INSERT OR IGNORE INTO cid10 (CID, CD_DESCRICAO) VALUES ('A00', 'Cólera');",
    "INSERT OR IGNORE INTO cid10 (CID, CD_DESCRICAO) VALUES ('J11', 'Influenza');",
    "INSERT OR IGNORE INTO cid10 (CID, CD_DESCRICAO) VALUES ('I10', 'Hipertensão essencial (primária)');",
    "INSERT OR IGNORE INTO cid10 (CID, CD_DESCRICAO) VALUES ('E11', 'Diabetes mellitus não-insulinodependente');",
    "INSERT OR IGNORE INTO cid10 (CID, CD_DESCRICAO) VALUES ('A09', 'Diarreia e gastroenterite de origem infecciosa presumível');",
    "INSERT OR IGNORE INTO cid10 (CID, CD_DESCRICAO) VALUES ('A15', 'Tuberculose respiratória');", # Para consulta A1
    "INSERT OR IGNORE INTO cid10 (CID, CD_DESCRICAO) VALUES ('I21', 'Infarto agudo do miocárdio');",
    "INSERT OR IGNORE INTO cid10 (CID, CD_DESCRICAO) VALUES ('C34', 'Neoplasia maligna dos pulmões');",
    "INSERT OR IGNORE INTO cid10 (CID, CD_DESCRICAO) VALUES ('J44', 'Doenças pulmonares obstrutivas crônicas');",
    "INSERT OR IGNORE INTO cid10 (CID, CD_DESCRICAO) VALUES ('F03', 'Demência não especificada');",
    "INSERT OR IGNORE INTO cid10 (CID, CD_DESCRICAO) VALUES ('N18', 'Insuficiência renal crônica');",
    "INSERT OR IGNORE INTO cid10 (CID, CD_DESCRICAO) VALUES ('V89', 'Acidente de trânsito');",
    "INSERT OR IGNORE INTO cid10 (CID, CD_DESCRICAO) VALUES ('K85', 'Apêndice e hérnias');",
    
    # procedimentos
    "INSERT OR IGNORE INTO procedimentos (PROC_REA, NOME_PROC) VALUES ('0303010037', 'Tratamento de pneumonia');",
    "INSERT OR IGNORE INTO procedimentos (PROC_REA, NOME_PROC) VALUES ('0401010023', 'Cirurgia cardiovascular');",
    "INSERT OR IGNORE INTO procedimentos (PROC_REA, NOME_PROC) VALUES ('0201010018', 'Biópsia simples');",
    "INSERT OR IGNORE INTO procedimentos (PROC_REA, NOME_PROC) VALUES ('0301010012', 'Atendimento de urgência');",
    "INSERT OR IGNORE INTO procedimentos (PROC_REA, NOME_PROC) VALUES ('101010010', 'Procedimento de teste');", # Para consulta A3
    "INSERT OR IGNORE INTO procedimentos (PROC_REA, NOME_PROC) VALUES ('0408050106', 'Tratamento de fraturas');",
    
    # municipios
    "INSERT OR IGNORE INTO municipios (codigo_6d, codigo_ibge, nome, latitude, longitude, estado) VALUES ('123456', '1234567', 'Município Exemplo 1', -23.5505, -46.6333, 'SP');",
    "INSERT OR IGNORE INTO municipios (codigo_6d, codigo_ibge, nome, latitude, longitude, estado) VALUES ('654321', '7654321', 'Município Exemplo 2', -22.9068, -43.1729, 'RJ');",
    "INSERT OR IGNORE INTO municipios (codigo_6d, codigo_ibge, nome, latitude, longitude, estado) VALUES ('111111', '1111111', 'Município Exemplo 3', -19.9208, -43.9378, 'MG');",
    "INSERT OR IGNORE INTO municipios (codigo_6d, codigo_ibge, nome, latitude, longitude, estado) VALUES ('431490', '4314902', 'Porto Alegre', -30.0346, -51.2177, 'RS');", # Para consultas A2 e B3
    "INSERT OR IGNORE INTO municipios (codigo_6d, codigo_ibge, nome, latitude, longitude, estado) VALUES ('430510', '4305108', 'Caxias do Sul', -29.1682, -51.1794, 'RS');", # Adicional para RS
    "INSERT OR IGNORE INTO municipios (codigo_6d, codigo_ibge, nome, latitude, longitude, estado) VALUES ('355030', '3550308', 'São Paulo', -23.5505, -46.6333, 'SP');",
    
    # dado_ibge
    "INSERT OR IGNORE INTO dado_ibge (codigo_municipio_completo, nome_municipio, sigla_estado, populacao, densidade_demografica, salario_medio, mortalidade_infantil) VALUES ('1234567', 'Município Exemplo 1', 'SP', 1000000, 50.0, 3000.0, 10.0);",
    "INSERT OR IGNORE INTO dado_ibge (codigo_municipio_completo, nome_municipio, sigla_estado, populacao, densidade_demografica, salario_medio, mortalidade_infantil) VALUES ('7654321', 'Município Exemplo 2', 'RJ', 500000, 40.0, 2500.0, 12.5);",
    "INSERT OR IGNORE INTO dado_ibge (codigo_municipio_completo, nome_municipio, sigla_estado, populacao, densidade_demografica, salario_medio, mortalidade_infantil) VALUES ('1111111', 'Município Exemplo 3', 'MG', 200000, 25.0, 2000.0, 15.0);",
    "INSERT OR IGNORE INTO dado_ibge (codigo_municipio_completo, nome_municipio, sigla_estado, populacao, densidade_demografica, salario_medio, mortalidade_infantil) VALUES ('4314902', 'Porto Alegre', 'RS', 1488000, 2900.0, 4000.0, 9.5);", # > 100k hab (C1) e para A5
    "INSERT OR IGNORE INTO dado_ibge (codigo_municipio_completo, nome_municipio, sigla_estado, populacao, densidade_demografica, salario_medio, mortalidade_infantil) VALUES ('4305108', 'Caxias do Sul', 'RS', 510906, 310.0, 3500.0, 8.0);",
    "INSERT OR IGNORE INTO dado_ibge (codigo_municipio_completo, nome_municipio, sigla_estado, populacao, densidade_demografica, salario_medio, mortalidade_infantil) VALUES ('3550308', 'São Paulo', 'SP', 12325232, 8000.0, 4500.0, 11.0);", # Maior população (B7)
    
    # internacoes (SEXO: 1=Masc, 2=Fem)
    "INSERT OR IGNORE INTO internacoes (N_AIH, CNES, DT_INTER, DT_SAIDA, QT_DIARIAS, PROC_REA, VAL_TOT, DIAG_PRINC, DIAG_SECUN, NASC, SEXO, IDADE, MUNIC_RES) VALUES ('987654321', '1234567', '2023-01-10', '2023-01-15', 5, '0303010037', 2500.00, 'J11', 'A00', '1980-05-15', 1, 42, '123456');",
    "INSERT OR IGNORE INTO internacoes (N_AIH, CNES, DT_INTER, DT_SAIDA, QT_DIARIAS, PROC_REA, VAL_TOT, DIAG_PRINC, DIAG_SECUN, NASC, SEXO, IDADE, MUNIC_RES) VALUES ('112233445', '7654321', '2023-02-20', '2023-02-28', 8, '0401010023', 15000.00, 'I10', 'E11', '1965-10-20', 2, 57, '654321');",
    "INSERT OR IGNORE INTO internacoes (N_AIH, CNES, DT_INTER, DT_SAIDA, QT_DIARIAS, PROC_REA, VAL_TOT, DIAG_PRINC, DIAG_SECUN, NASC, SEXO, IDADE, MUNIC_RES) VALUES ('556677889', '1112223', '2023-03-05', '2023-03-06', 1, '0301010012', 500.00, 'A09', '', '1995-12-05', 1, 27, '111111');",
    "INSERT OR IGNORE INTO internacoes (N_AIH, CNES, DT_INTER, DT_SAIDA, QT_DIARIAS, PROC_REA, VAL_TOT, DIAG_PRINC, DIAG_SECUN, NASC, SEXO, IDADE, MUNIC_RES) VALUES ('998877665', '1234567', '2023-04-10', '2023-04-18', 8, '0201010018', 3200.00, 'J11', '', '2010-02-18', 2, 13, '123456');",
    "INSERT OR IGNORE INTO internacoes (N_AIH, CNES, DT_INTER, DT_SAIDA, QT_DIARIAS, PROC_REA, VAL_TOT, DIAG_PRINC, DIAG_SECUN, NASC, SEXO, IDADE, MUNIC_RES) VALUES ('444555666', '2234416', '2023-05-01', '2023-05-05', 4, '101010010', 1000.00, 'A15', '', '1970-01-01', 2, 53, '431490');", # Para consulta A4 e hospital '2234416' e mulher 
    "INSERT OR IGNORE INTO internacoes (N_AIH, CNES, DT_INTER, DT_SAIDA, QT_DIARIAS, PROC_REA, VAL_TOT, DIAG_PRINC, DIAG_SECUN, NASC, SEXO, IDADE, MUNIC_RES) VALUES ('333222111', '2234416', '2023-01-01', '2023-02-10', 40, '0303010037', 15000.00, 'J11', '', '1950-01-01', 1, 73, '430510');", # Para consulta B5 (diarias > 30)
    "INSERT OR IGNORE INTO internacoes (N_AIH, CNES, DT_INTER, DT_SAIDA, QT_DIARIAS, PROC_REA, VAL_TOT, DIAG_PRINC, DIAG_SECUN, NASC, SEXO, IDADE, MUNIC_RES) VALUES ('101010101', '9999999', '2023-06-01', '2023-06-05', 4, '0408050106', 2000.00, 'V89', '', '1990-01-01', 1, 33, '355030');",
    "INSERT OR IGNORE INTO internacoes (N_AIH, CNES, DT_INTER, DT_SAIDA, QT_DIARIAS, PROC_REA, VAL_TOT, DIAG_PRINC, DIAG_SECUN, NASC, SEXO, IDADE, MUNIC_RES) VALUES ('202020202', '9999999', '2023-07-01', '2023-07-05', 4, '0301010012', 2000.00, 'I21', '', '1960-01-01', 2, 63, '355030');",
    "INSERT OR IGNORE INTO internacoes (N_AIH, CNES, DT_INTER, DT_SAIDA, QT_DIARIAS, PROC_REA, VAL_TOT, DIAG_PRINC, DIAG_SECUN, NASC, SEXO, IDADE, MUNIC_RES) VALUES ('303030303', '1234567', '2023-08-01', '2023-08-05', 4, '0301010012', 2000.00, 'C34', '', '1955-01-01', 2, 68, '123456');",
    "INSERT OR IGNORE INTO internacoes (N_AIH, CNES, DT_INTER, DT_SAIDA, QT_DIARIAS, PROC_REA, VAL_TOT, DIAG_PRINC, DIAG_SECUN, NASC, SEXO, IDADE, MUNIC_RES) VALUES ('404040404', '7654321', '2023-09-01', '2023-09-05', 4, '0301010012', 2000.00, 'J44', '', '1940-01-01', 1, 83, '654321');",
    "INSERT OR IGNORE INTO internacoes (N_AIH, CNES, DT_INTER, DT_SAIDA, QT_DIARIAS, PROC_REA, VAL_TOT, DIAG_PRINC, DIAG_SECUN, NASC, SEXO, IDADE, MUNIC_RES) VALUES ('505050505', '1112223', '2023-10-01', '2023-10-05', 4, '0301010012', 2000.00, 'F03', '', '1930-01-01', 2, 93, '111111');",
    "INSERT OR IGNORE INTO internacoes (N_AIH, CNES, DT_INTER, DT_SAIDA, QT_DIARIAS, PROC_REA, VAL_TOT, DIAG_PRINC, DIAG_SECUN, NASC, SEXO, IDADE, MUNIC_RES) VALUES ('606060606', '2234416', '2023-11-01', '2023-11-05', 4, '0301010012', 2000.00, 'N18', '', '1950-01-01', 2, 73, '431490');",
    "INSERT OR IGNORE INTO internacoes (N_AIH, CNES, DT_INTER, DT_SAIDA, QT_DIARIAS, PROC_REA, VAL_TOT, DIAG_PRINC, DIAG_SECUN, NASC, SEXO, IDADE, MUNIC_RES) VALUES ('707070707', '1234567', '2023-12-01', '2023-12-05', 4, '0301010012', 2000.00, 'K85', '', '1985-01-01', 1, 38, '123456');",
    
    # mortes
    "INSERT OR IGNORE INTO mortes (N_AIH, CID_MORTE) VALUES ('987654321', 'J11');",
    "INSERT OR IGNORE INTO mortes (N_AIH, CID_MORTE) VALUES ('112233445', 'I10');", # Ocorreu com mulher, para consulta C3
    "INSERT OR IGNORE INTO mortes (N_AIH, CID_MORTE) VALUES ('998877665', 'J11');", # Ocorreu com mulher, para consulta C3
    "INSERT OR IGNORE INTO mortes (N_AIH, CID_MORTE) VALUES ('444555666', 'A15');", # Ocorreu com mulher, para consulta C3
    "INSERT OR IGNORE INTO mortes (N_AIH, CID_MORTE) VALUES ('101010101', 'V89');",
    "INSERT OR IGNORE INTO mortes (N_AIH, CID_MORTE) VALUES ('202020202', 'I21');", # Ocorreu com mulher
    "INSERT OR IGNORE INTO mortes (N_AIH, CID_MORTE) VALUES ('303030303', 'C34');", # Ocorreu com mulher
    "INSERT OR IGNORE INTO mortes (N_AIH, CID_MORTE) VALUES ('404040404', 'J44');",
    "INSERT OR IGNORE INTO mortes (N_AIH, CID_MORTE) VALUES ('505050505', 'F03');", # Ocorreu com mulher
    "INSERT OR IGNORE INTO mortes (N_AIH, CID_MORTE) VALUES ('606060606', 'N18');", # Ocorreu com mulher
    "INSERT OR IGNORE INTO mortes (N_AIH, CID_MORTE) VALUES ('707070707', 'K85');",
    
    # condicoes_especificas
    "INSERT OR IGNORE INTO condicoes_especificas (N_AIH, IND_VDRL) VALUES ('987654321', 'Negativo');",
    "INSERT OR IGNORE INTO condicoes_especificas (N_AIH, IND_VDRL) VALUES ('112233445', 'Positivo');", # Para consulta B4
    "INSERT OR IGNORE INTO condicoes_especificas (N_AIH, IND_VDRL) VALUES ('556677889', 'Negativo');",
    "INSERT OR IGNORE INTO condicoes_especificas (N_AIH, IND_VDRL) VALUES ('444555666', 'Positivo');" # Para consulta B4
]

with sqlite3.connect("database.db") as db:
    cursor = db.cursor()
    cursor.execute("PRAGMA foreign_keys = ON;")
    for q in insert_queries:
        cursor.execute(q)
    db.commit()
    print("Muitos e muitos dados inseridos com sucesso!")

Muitos e muitos dados inseridos com sucesso!


5.1 Consultas em Álgebra Relacional

A1. πCD_DESCRICAO
(σCID='A15'
(cid10))
A2. πlatitude, longitude
(σnome='Porto Alegre'
(municipios))
A3. πNOME_PROC
(σPROC_REA='101010010'
(procedimentos))
A4. πN_AIH, CD_DESCRICAO
(σCNES='2234416'
(internacoes) ▷◁DIAG_PRINC=CID 
cid10)
A5. πnome, populacao
(σestado='RS'
(municipios) ▷◁codigo_ibge=codigo_municipio_completo 
dado_ibge)


In [ ]:
queryA1 = "SELECT CD_DESCRICAO FROM cid10 WHERE CID = 'A15';"
queryA2 = """SELECT latitude, longitude
FROM municipios
WHERE nome = 'Porto Alegre'"""

queryA3 = """SELECT NOME_PROC
FROM procedimentos
where PROC_REA = '101010010';"""
queryA4 = """SELECT N_AIH, CD_DESCRICAO
FROM internacoes
JOIN cid10 ON DIAG_PRINC = CID
WHERE CNES = '2234416';"""
queryA5 = """SELECT nome, populacao
FROM municipios
JOIN dado_ibge on codigo_ibge = codigo_municipio_completo
WHERE estado = 'RS';"""

queryb1 = """SELECT COUNT(*)
FROM internacoes"""

queryb2 = """SELECT AVG(IDADE)
FROM internacoes"""

queryb3 = """SELECT COUNT(*)
FROM municipios
WHERE estado = 'RS'"""
queryb4 = """SELECT COUNT(*)
FROM condicoes_especificas
WHERE IND_VDRL = '1'"""
queryb5 = """SELECT COUNT(*)
FROM internacoes
WHERE QT_DIARIAS > 30"""
queryb6 = """SELECT NATUREZA, GESTAO, NAT_JUR
FROM hospital
where CNES = '2234416';"""
queryb7 = """SELECT nome_municipio
FROM dado_ibge
ORDER BY populacao DESC LIMIT 1"""
queryb8 = """SELECT DIAG_PRINC, count(*)
FROM internacoes
GROUP BY DIAG_PRINC
ORDER BY count(*) DESC LIMIT 1"""
queryb9 = """SELECT COUNT(*)
FROM mortes"""
queryb10 = """SELECT CID, CD_DESCRICAO, count(*)
FROM cid10
JOIN mortes ON CID_MORTE = CID
GROUP BY CID
ORDER BY count(*) DESC LIMIT 10"""



consultas

In [83]:
# Usando a biblioteca Pandas para visualizar o resultado como uma tabela bonita
df_municipios = pd.read_sql_query(queryb10, db)
display(df_municipios)

# Como alternativa, você pode usar apenas o sqlite3 puro (mas fica em formato de lista):
# cursor.execute(query)
# resultados = cursor.fetchall()
# print(resultados)

,CID,CD_DESCRICAO,count(*)
0,J11,Influenza,2
1,V89,Acidente de trânsito,1
2,N18,Insuficiência renal crônica,1
3,K85,Apêndice e hérnias,1
4,J44,Doenças pulmonares obstrutivas crônicas,1
5,I21,Infarto agudo do miocárdio,1
6,I10,Hipertensão essencial (primária),1
7,F03,Demência não especificada,1
8,C34,Neoplasia maligna dos pulmões,1
9,A15,Tuberculose respiratória,1
